In [1]:
from assetextractor.extraction.utils import Config

from assetextractor.parsing.core.assets import Asset, AssetCache
from assetextractor.parsing.core.templates import Template
from assetextractor.parsing.core.attributes import ListAttribute

import csv
from pathlib import Path
import json
from lxml.etree import tostring

import typing as t

In [2]:
config = Config.from_json("config.json")
assets = AssetCache.load(config)
templates = assets.templates

In [3]:
params_path = Path("../anno-117-calculator/js/params.js")
with params_path.open(encoding="utf-8") as f:
    js_content = f.read()

prefix = 'if(window.params == null)window.params='
if not js_content.startswith(prefix):
    raise ValueError("params.js does not start with the expected prefix")
json_str = js_content[len(prefix):].strip()
if json_str.endswith(';'):
    json_str = json_str[:-1]

try:
    params = json.loads(json_str)
except Exception as e:
    raise ValueError(f"Failed to parse params.js as JSON: {e}")

product_to_factory = {}
product_to_inputs = {}
factory_to_upkeep = {}
factory_tpmin = {}
product_to_price = {}

for factory in params.get("factories", []):
    guid = factory["guid"]
    if factory.get("outputs") is not None:
        product = factory.get("outputs")[0]["Product"]
        product_to_inputs[product] = [(entry["Product"], entry["Amount"]) for entry in factory.get("inputs", [])]
        factory_to_upkeep[guid] = factory["maintenances"][0]["Amount"]
        factory_tpmin[guid] = factory["tpmin"]

for prod in params.get("products", []):
    try:
        guid = int(prod["guid"])
        product_to_price[guid] = assets[guid].Product.BasePrice()
        if prod.get("mainFactory") is not None:
            main_factory = int(prod["mainFactory"])
            product_to_factory[guid] = main_factory
    except (KeyError, ValueError, TypeError):
        raise ValueError(f"Invalid product entry: {prod}")
    


In [4]:
def get_production_price(guid):
    factory = product_to_factory.get(guid)

    if factory is None:
        return None

    return factory_to_upkeep[factory] / factory_tpmin[factory]+ sum(get_production_price(input[0]) * input[1] for input in product_to_inputs[guid])

In [5]:
factory = product_to_factory.get(2145)

In [9]:
len(product_to_inputs)

110

In [ ]:
product_to_inputs[factory]

KeyError: 3191

In [ ]:
get_production_price(2145)

160.0

In [7]:
attributes = assets.datasets["NeedAttributeType"].literals

In [14]:
rows = [["name"] + attributes + [f"Area Buff {attribute}" for attribute in attributes]]
empty_attributes = [None for i in range(len(attributes))]

for need in templates["Need"].assets:
    product = need.Need.NeedProduct()
    row = [str(product)] #name
    for attribute in attributes:
        val = need.find(f"Need.NeedAttributes.{attribute}.Value").value
        row.append(None if val is None or val == 0 else val)

    if product.guid in product_to_factory:
        attribute_values = {}
        for attribute in attributes:
            attribute_values[attribute] = 0

        for entry in assets.get(product_to_factory[product.guid]).Building.FunctionalEffects:
            buff = entry.FunctionalEffect()      

            area_attributes = buff.Effect.Buffs[0].GUID().BuildingUpgrade.AdditionalAttributes
            for attribute in attributes:
                val = area_attributes[attribute].AmountOrPercent.value
                if val is not None:
                    attribute_values[attribute] += val
        
        if len(row) == 1:
            row += empty_attributes
        else:
            row += [None if val == 0 else val for attr, val in attribute_values.items()]
    else:
        row += empty_attributes + empty_attributes

    rows.append(row)

In [15]:


output_path = Path("results/tables/needs.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", newline='', encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(rows)

In [12]:
output_path = Path("results/tables/products.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", newline='', encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["guid", "product", "production price", "base price"])
    for product in product_to_price:
        writer.writerow([product, str(assets[product]), get_production_price(product), product_to_price[product]])